# Scenario comparison

Compares month-end **nominal** net worth across any number of scenarios against a
baseline, and finds each variant's break-even month — the point after which it
stays ahead for good.

Pick a baseline and up to 4 variant scenarios below — every chart and table in
this notebook updates automatically as the selection changes, no re-running
required. Each must already have a run (`analysis sim load <name> && analysis
sim run <name>`); the newest run for each is used.

> **Note:** after running cells, `git status` will show this notebook as modified
> even with no real changes — see [notebooks/README.md](./README.md#git-status-shows-a-notebook-as-modified-after-running-it).



In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

from IPython.display import display

from analysis.sim.loader import list_scenario_names
from analysis.sim.notebook_ui.pickers import multi_scenario_picker
from analysis.sim.notebook_ui.compare_plots import (
    comparability_widget, rules_and_events_widget,
    net_worth_compare_widget, breakeven_widget,
    year_end_widget,
)


REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = Path(os.environ.get("ANALYSIS_DATA_DIR", REPO / "data" / "shared" / "sim"))
MAX_VARIANTS = 4

_available = list_scenario_names(DATA_DIR)
_default_baseline = "current" if "current" in _available else _available[0]
_default_variants = [name for name in _available if name != _default_baseline][:MAX_VARIANTS]

# CONNECTIONS holds the live RunConnections, refreshed in place whenever either picker changes.
PICKER, CONNECTIONS = multi_scenario_picker(
    DATA_DIR, _default_baseline, _default_variants, MAX_VARIANTS
)
PICKER

## Are these runs comparable?

A comparison is only meaningful if both runs came from the same code and the same
data version — otherwise a difference here could be an unrelated edit rather than
the scenario change under test.


In [ ]:
CONNECTIONS.show(comparability_widget)

## What actually differs?

Each scenario inherits everything from its parent except the rows it overrides, so
this is the whole mechanism behind any difference in the charts below. If a rule
you expected to change isn't listed, the override didn't take effect.

Two sources of difference are shown separately:

- **Rules** — as they stand at the *end* of the run. A calendar event that edits a
  rule mid-run (`set_amount`) shows up here as a changed amount, because the run
  snapshot records the post-event state.
- **Calendar events** — one-time interventions, including the benefit cut.


In [ ]:
CONNECTIONS.show(rules_and_events_widget)

## Net worth over time

Nominal dollars. (`net_worth_constant_cpi` is available in each run if you want
the inflation-adjusted view later.)


In [ ]:
CONNECTIONS.show(net_worth_compare_widget)

## Break-even

**Break-even** is the first month after which the variant stays ahead for the rest
of the scenario — not merely the first month it pokes above zero, which can happen
transiently when a lumpy payment lands in one scenario before the other.


In [ ]:
CONNECTIONS.show(breakeven_widget)

In [ ]:
CONNECTIONS.show(year_end_widget)